In [1]:
import matplotlib
matplotlib.use("Agg")
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
import h5py

import importlib
import os
import sys
import re
import glob

In [2]:
path_openket = "//home//ultrxvioletx//GitHub//openket"
if path_openket not in sys.path:
    sys.path.append(path_openket)
# elimina el caché de func.py para que no lea los datos anteriores
if 'func' in sys.modules:
    del sys.modules['func']
if os.path.exists('func.py'):
    os.remove('func.py')

from openket.core.diracobject import Ket, Bra, Operator, AnnihilationOperator, CreationOperator
from openket.core.evolution import build_ode, sym2num, init_state
from openket.core.metrics import dag, comm, ptrace, trace, normalize, sub_qexpr, op2dict, qmatrix

import style
importlib.reload(style)
from style import *
set_style()

2026-04-02 15:02:48,050 - openket - INFO - openket v0.1.0 initialized successfully.


##### funciones auxiliares

In [3]:
def convergencia(tiempo, fotones, tol=1e-3, porcentaje=0.1):
    npoints = int(len(tiempo) * porcentaje)
    if npoints < 2:
        return False, np.nan # no hay suficientes puntos
    fotones = fotones[-npoints:]
    tiempo = tiempo[-npoints:]
    df = np.gradient(fotones, tiempo)
    df_mean = np.mean(np.abs(df))
    return df_mean < tol, df_mean

In [4]:
def procesafile(at, lvl, folder, barrido="scan"):
    global detunings, Nss, non_converged, Probs, niveles, attrs
    detunings = []
    Probs = [[] for _ in np.arange(lvl**at)]
    Nss = []
    attrs = {}
    
    non_converged = []
    files = glob.glob(os.path.join(path,'*.h5'))
    for i,file in enumerate(files):
        with h5py.File(file, 'r') as f:
            dataset = os.path.basename(file).replace('.h5', '')
            if i == 0:
                attrs = dict(f[dataset].attrs)
            
            detuning = f[dataset].attrs[barrido]
            tt = f[dataset].attrs['t']
            niveles = f[dataset].attrs['probs_label']
            tiempo = np.linspace(float(tt[0]), float(tt[1]), int(tt[2]))
    
            if dataset not in f:
                print(f"  no se encontró el dataset '{dataset}' en el archivo. Saltando.")
                continue

            detunings.append(detuning)
            rho = f[dataset][:]
            lvl_prob = [rho[:, i] for i in np.arange(lvl**at)] # las primeras n columnas son las probabilidades
            N_expect = rho[:, lvl**at] # la última columna es el valor medio de fotones en la cavidad
            N_expect = np.real(N_expect)

            # promediamos el último 25% de los puntos
            npoints = int(rho.shape[0] * 0.25)
            Nss.append(np.mean(N_expect[-npoints:]))
            for i,ele in enumerate(lvl_prob):
                Probs[i].append(np.mean(ele[-npoints:]))
            
            is_converged, derivative = convergencia(tiempo, N_expect)
            if not is_converged:
                non_converged.append([detuning, np.mean(N_expect[-npoints:])])
                
    # ordena por detuning para mejorar la gráfica
    detunings = np.array(detunings)
    sorti = np.argsort(detunings)
    
    detunings = detunings[sorti]
    Nss = np.array(Nss)[sorti]
    Probs = [np.real(np.array(p))[sorti] for p in Probs]

In [5]:
def picos(x,y):
    x = np.array(x)
    y = np.array(y)
    peaks, _ = find_peaks(y, prominence=0.001)

    if len(peaks) < 2:
        return None, None
    elif len(peaks) == 2:
        x1, x2 = x[peaks[0]], x[peaks[1]]
    else:
        # 3+ picos: tomar los dos extremos en x (ignorar el del medio)
        x1, x2 = x[peaks[0]], x[peaks[-1]]

    return x1, x2

## solo átomo

In [6]:
at,lvl = 1,4
file = os.path.join("asaf",f"{at}at{lvl}lvl",f"{at}at{lvl}lvl.h5")
with h5py.File(file, 'r') as f:
    dataset = os.path.basename(file).replace('.h5', '')
    attrs = dict(f[dataset].attrs)
    
    tt = f[dataset].attrs['t']
    niveles = f[dataset].attrs['probs_label']
    tiempo = np.linspace(float(tt[0]), float(tt[1]), int(tt[2]))

    rho = f[dataset][:]
    lvl_prob = [rho[:, i] for i in np.arange(lvl**at)] # las primeras n columnas son las probabilidades
plvls = ["e", "s"]
step = 20
for j,plvl in enumerate(plvls):
    pattern_str = plvl.replace("_", ".")
    pattern_str = pattern_str[:at] # recorta al largo del dataset
    pattern = re.compile(f"^{pattern_str}$")
    indices = [i for i, lbl in enumerate(niveles) if pattern.search(lbl)]
    prob = np.sum([lvl_prob[i] for i in indices], axis=0)
    
    plt.plot(tiempo[::step], prob[::step], "s-", label=rf"${latex["P"]}{plvl}$", markersize=5, linewidth=1, color=colores["estados"][j])
plt.axhline(0, color="gray", linestyle="--", linewidth=1)
plt.axhline(1, color="gray", linestyle="--", linewidth=1)
plt.xlabel(r"tiempo $\kappa ^{-1}$")
plt.ylabel(r"Probabilidad de excitación")
plt.legend()
ax = plt.gca()
format_ax(ax, xstep=50)
plt.savefig(f"../figuras/1at4lvl.png")
plt.close()

In [7]:
attrs

{'detuning_c': np.float64(-99.0),
 'detuning_p': np.float64(100.000625),
 'gamma_12': np.float64(0.0),
 'gamma_e': np.float64(0.0),
 'gamma_s': np.float64(0.0),
 'idx': np.int32(0),
 'kappa': np.float64(1.0),
 'omega_DR': np.float64(0.0),
 'omega_EE': np.float64(0.1),
 'probs_label': array(['g', 'e', 's', 'p'], dtype=object),
 'rabi_c': np.float64(20.0),
 'rabi_p': np.float64(0.5),
 'scan': np.float64(0.0),
 'stark_g': np.float64(0.000625),
 'stark_s': np.float64(1.0),
 't': array(['0', '200.0', '1000'], dtype=object)}

## átomo + cavidad

In [22]:
at,lvl = 1,4
label = "cavidad"
path = os.path.join("asaf",f"{at}at{lvl}lvl_{label}")

procesafile(at, lvl, path)

mask = (detunings >= -4) & (detunings <= 7)
plt.plot(detunings[mask], Nss[mask], "s-", markersize=5, linewidth=1, color=colores["fotones"])
if non_converged:
    dtn = [ele[0] for ele in non_converged]
    nmean = [ele[1] for ele in non_converged]
    plt.scatter(dtn, nmean, color="red")
plt.axvline(1, color="gray", linestyle="--", linewidth=1, alpha=0.5)
x1,x2 = picos(detunings,Nss)
ax = plt.gca()
segmento(ax, x1, x2, y=max(Nss))
#plt.xlabel(rf"${latex["Dpa"]}$")
plt.ylabel(rf"Número medio de fotones ${latex["Nss"]}$")
format_ax(ax, xstep=1, ystep=0.2, ymax=1)
plt.savefig(f"../figuras/1at4lvl_fotones.png")
plt.close()
fdist_1at = x2-x1
print(x1,x2)

plvls = ["e", "s"]
for j,plvl in enumerate(plvls):
    pattern_str = plvl.replace("_", ".")
    pattern_str = pattern_str[:at] # recorta al largo del dataset
    pattern = re.compile(f"^{pattern_str}$")
    indices = [i for i, lbl in enumerate(niveles) if pattern.search(lbl)]
    prob = np.sum([Probs[i] for i in indices], axis=0)
    
    plt.plot(detunings[mask], prob[mask], "s-", label=rf"${latex["P"]}{plvl}$", markersize=5, linewidth=1, color=colores["estados"][j])
plt.axvline(1, color="gray", linestyle="--", linewidth=1, alpha=0.5)
x1, x2 = picos(detunings,prob)
ax = plt.gca()
segmento(ax, x1, x2, y=max(prob))
plt.xlabel(rf"${latex["scan"]}$")
plt.ylabel(r"Probabilidad de excitación")
ax = plt.gca()
format_ax(ax, xstep=1, ystep=0.2, ymax=1)
plt.legend()
plt.savefig(f"../figuras/1at4lvl_excitado.png")
plt.close()
edist_1at = x2-x1
print(x1,x2)

-0.3797 2.6835
-0.3797 2.4051


In [9]:
attrs

{'detuning_c': np.float64(-99.0),
 'detuning_p': np.float64(108.237025),
 'gamma_12': np.float64(0.0),
 'gamma_e': np.float64(1.0),
 'gamma_s': np.float64(0.0),
 'idx': np.int32(53),
 'kappa': np.float64(1.0),
 'omega_DR': np.float64(0.0),
 'omega_EE': np.float64(0.1),
 'probs_label': array(['g', 'e', 's', 'p'], dtype=object),
 'rabi_c': np.float64(20.0),
 'rabi_p': np.float64(0.5),
 'scan': np.float64(8.2364),
 'stark_g': np.float64(0.000625),
 'stark_s': np.float64(1.0),
 't': array(['0', '200.0', '1000'], dtype=object)}

## 2 átomos + cavidad

In [33]:
at,lvl = 2,4
label = "cavidad"
path = os.path.join("asaf",f"{at}at{lvl}lvl_{label}")

procesafile(at, lvl, path)

mask = (detunings >= -3) & (detunings <= 8)
plt.plot(detunings[mask], Nss[mask], "s-", markersize=5, linewidth=1, color=colores["fotones"])
if non_converged:
    dtn = [ele[0] for ele in non_converged]
    nmean = [ele[1] for ele in non_converged]
    plt.scatter(dtn, nmean, color="red")
#plt.axvline(1, color="gray", linestyle="--", linewidth=1, alpha=0.5)
x1, x2 = picos(detunings,Nss)
ax = plt.gca()
segmento(ax, x1, x2, y=max(Nss))
#plt.xlabel(rf"${latex["Dpa"]}$")
plt.ylabel(rf"Número medio de fotones ${latex["Nss"]}$")
ax = plt.gca()
format_ax(ax, xstep=1, ystep=0.2, ymax=1)
plt.savefig(f"../figuras/2at4lvl_fotones0.png")
plt.close()
fdist_2at = x2-x1
print(x1,x2)

plvls = ["s.", "ss"]
labels = ["P_{gs}+P_{sg}", "P_{ss}"]
for j,plvl in enumerate(plvls):
    pattern_str = plvl.replace("_", ".")
    pattern_str = pattern_str[:at] # recorta al largo del dataset
    pattern = re.compile(f"^{pattern_str}$")
    indices = [i for i, lbl in enumerate(niveles) if pattern.search(lbl)]
    prob = np.sum([Probs[i] for i in indices], axis=0)
    if j==0: prob_s = prob
    
    plt.plot(detunings[mask], prob[mask], "s-", label=rf"${labels[j]}$", markersize=5, linewidth=1, color=colores["estados"][j+1])
#plt.axvline(1, color="gray", linestyle="--", linewidth=1, alpha=0.5)
x1, x2 = picos(detunings,prob_s)
ax = plt.gca()
segmento(ax, x1, x2, y=max(prob_s))
plt.xlabel(rf"${latex["scan"]}$")
plt.ylabel(r"Probabilidad de excitación")
ax = plt.gca()
format_ax(ax, xstep=1, ystep=0.2, ymax=1)
plt.legend()
plt.savefig(f"../figuras/2at4lvl_excitado0.png")
plt.close()
edist_2at = x2-x1
print(x1,x2)

-0.7722 4.9367
-0.7722 4.7975


In [11]:
attrs

{'detuning_c': np.float64(-99.0),
 'detuning_p': np.float64(101.364225),
 'gamma_12': np.float64(0.0),
 'gamma_e': np.float64(1.0),
 'gamma_s': np.float64(0.0),
 'idx': np.int32(26),
 'kappa': np.float64(1.0),
 'omega_DR': np.float64(0.0),
 'omega_EE': np.float64(0.0),
 'probs_label': array(['gg', 'ge', 'gs', 'gp', 'eg', 'ee', 'es', 'ep', 'sg', 'se', 'ss',
        'sp', 'pg', 'pe', 'ps', 'pp'], dtype=object),
 'rabi_c': np.float64(20.0),
 'rabi_p': np.float64(0.5),
 'scan': np.float64(1.3636),
 'stark_g': np.float64(0.000625),
 'stark_s': np.float64(1.0),
 't': array(['0', '200.0', '1000'], dtype=object)}

In [12]:
print("1at=",fdist_1at, "2at=",fdist_2at, "factor=",fdist_2at/fdist_1at)
print(edist_2at/edist_1at)

1at= 3.0546 2at= 5.6 factor= 1.8333005958226933
1.9090714285714283


## 2 átomos + cavidad + bloqueo

In [73]:
def picos(x,y):
    dist = 3
    x = np.array(x)
    y = np.array(y)
    peaks, _ = find_peaks(y, prominence=0.01)

    # filtrar picos pequeños
    threshold = max(y[peaks]) * 0.1
    peaks = peaks[y[peaks] >= threshold]

    neg = peaks[x[peaks] < 0]
    pos = peaks[x[peaks] > 0]

    if len(neg) == 0 or len(pos) == 0:
        return None, (None, None)

    # el más alto del lado negativo
    p_neg = neg[np.argmax(y[neg])]
    x1 = x[p_neg]

    # del lado positivo: ordenar por altura y tomar el primero que cumpla min_dist
    pos_by_height = pos[np.argsort(y[pos])[::-1]]
    p_pos = next((p for p in pos_by_height if x[p] - x1 >= dist), None)

    if p_pos is None:
        return None, (None, None)

    x2 = x[p_pos]
    return x1, x2

In [81]:
at,lvl = 2,4
bloqueos = [0.1, 1, 5, 10, 15, 20, 50]
mask = (detunings >= -3) & (detunings <= 8)
for bloqueo in bloqueos:
    bstr = str.replace(str(bloqueo),".","")
    label = f"bloqueo{bstr}"
    path = os.path.join("asaf",f"{at}at{lvl}lvl_{label}")
    
    procesafile(at, lvl, path)
    
    plt.plot(detunings[mask], Nss[mask], "s-", markersize=5, linewidth=1, color=colores["fotones"])
    if non_converged:
        dtn = [ele[0] for ele in non_converged]
        nmean = [ele[1] for ele in non_converged]
        plt.scatter(dtn, nmean, color="red")
    #plt.axvline(1, color="gray", linestyle="--", linewidth=1, alpha=0.5)
    x1, x2 = picos(detunings,Nss)
    ax = plt.gca()
    segmento(ax, x1, x2, y=max(Nss))
    if bloqueo==50:
        plt.xlabel(rf"${latex["scan"]}$")
    plt.ylabel(rf"Número medio de fotones ${latex["Nss"]}$")
    ax = plt.gca()
    format_ax(ax, xstep=1, ystep=0.2, ymax=0.6)
    plt.text(0.05, 0.75, rf"${latex["Oee"]} = {bloqueo:.1f}$",transform=ax.transAxes, va="top", ha="left", fontsize=12)
    plt.savefig(f"../figuras/2at4lvl_fotones{bstr}.png")
    plt.close()
    fdist_2atb = x2-x1
    
    plvls = ["s.", "ss"]
    labels = ["P_{gs}+P_{sg}", "P_{ss}"]
    for j,plvl in enumerate(plvls):
        pattern_str = plvl.replace("_", ".")
        pattern_str = pattern_str[:at] # recorta al largo del dataset
        pattern = re.compile(f"^{pattern_str}$")
        indices = [i for i, lbl in enumerate(niveles) if pattern.search(lbl)]
        prob = np.sum([Probs[i] for i in indices], axis=0)
        if j==0: prob_s = prob
        
        plt.plot(detunings[mask], prob[mask], "s-", label=rf"${labels[j]}$", markersize=5, linewidth=1, color=colores["estados"][j+1])
    #plt.axvline(1, color="gray", linestyle="--", linewidth=1, alpha=0.5)
    x1, x2 = picos(detunings,prob_s)
    ax = plt.gca()
    segmento(ax, x1, x2, y=max(prob_s))
    if bloqueo==50:
        plt.xlabel(rf"${latex["scan"]}$")
    plt.ylabel(r"Probabilidad de excitación")
    ax = plt.gca()
    format_ax(ax, xstep=1, ystep=0.2, ymax=0.6)
    plt.text(0.75, 0.75, rf"${latex["Oee"]} = {bloqueo:.1f}$",transform=ax.transAxes, va="top", ha="left", fontsize=12)
    plt.legend()
    plt.savefig(f"../figuras/2at4lvl_excitado{bstr}.png")
    plt.close()
    print(x1,x2)

-0.7722 4.7975
-0.4937 4.7975
-0.7722 4.7975
-0.7722 4.6582
-0.7722 7.0253
-0.7722 4.6582
-0.7722 4.6582


In [14]:
x1, x2

(np.float64(-0.9273), np.float64(4.6727))

## omegas

In [79]:
at,lvl = 2,4
path = os.path.join("omegas",f"{at}at{lvl}lvl")

procesafile(at, lvl, path, barrido="omega_EE")

cut = 20
mask = (detunings >= -cut) & (detunings <= cut)
plt.plot(detunings[mask], Nss[mask], "s-", markersize=5, linewidth=1, color=colores["fotones"])
if non_converged:
    dtn = [ele[0] for ele in non_converged]
    nmean = [ele[1] for ele in non_converged]
    plt.scatter(dtn, nmean, color="red")
plt.axvline(0, color="gray", linestyle="--", linewidth=1)
plt.xlabel(rf"${latex["Oee"]}$")
plt.ylabel(rf"Número medio de fotones ${latex["Nss"]}$")
#ax = plt.gca()
#format_ax(ax, xstep=1, ystep=0.01, ymax=0.03)
plt.savefig(f"../figuras/fig:omegas_2at4lvl_fotones.png")
plt.close()

niveles = ["gg","ge","gs","gp","eg","ee","es","ep","sg","se","ss","sp","pg","pe","ps","pp"]
idx = {lbl: i for i, lbl in enumerate(niveles)}

prob_ss = Probs[idx["ss"]]
prob_xor = Probs[idx["sg"]] + Probs[idx["gs"]] + Probs[idx["se"]] + Probs[idx["es"]] + Probs[idx["sp"]] + Probs[idx["ps"]]

plt.plot(detunings[mask], prob_xor[mask], "s-", label=r"$P_{gs}+P_{sg}$", markersize=5, linewidth=1, color=colores["estados"][1])
plt.plot(detunings[mask], prob_ss[mask],  "s-", label=r"$P_{ss}$",         markersize=5, linewidth=1, color=colores["estados"][2])
plt.xlabel(rf"${latex["Oee"]}$")
plt.ylabel(r"Probabilidad de excitación")
#ax = plt.gca()
#format_ax(ax, xstep=2)
plt.legend()
plt.savefig(f"../figuras/fig:omegas_2at4lvl_excitado.png")
plt.close()

##### aux